In [0]:
catalog, schema = "workspace", "dm2_project"

# Clean transactions: drop returns/cancellations, compute revenue
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.{schema}.silver_sales AS
SELECT
  CAST(Invoice AS STRING)                         AS invoice_no,
  CAST(StockCode AS STRING)                       AS stock_code,
  Description                                      AS description,
  CAST(Quantity AS INT)                           AS quantity,
  CAST(Price AS DOUBLE)                            AS unit_price,
  CAST(Quantity AS INT) * CAST(Price AS DOUBLE)   AS line_revenue,
  to_timestamp(InvoiceDate)                       AS invoice_ts,
  date_trunc('MONTH', to_timestamp(InvoiceDate))  AS invoice_month,
  Customer_ID                                      AS customer_id,
  Country                                          AS country
FROM {catalog}.{schema}.bronze_sales
WHERE Quantity > 0 AND Price > 0
  AND Invoice IS NOT NULL
  AND Invoice NOT LIKE 'C%'
""")

# Curate country names, then join bronze_country -> region
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.{schema}.silver_sales_enriched AS
WITH s AS (
  SELECT *,
    CASE country
      WHEN 'EIRE'            THEN 'Ireland'
      WHEN 'USA'             THEN 'United States'
      WHEN 'RSA'             THEN 'South Africa'
      WHEN 'Channel Islands' THEN 'United Kingdom'
      WHEN 'Unspecified'     THEN NULL
      ELSE country
    END AS country_clean
  FROM {catalog}.{schema}.silver_sales
)
SELECT s.*, COALESCE(c.Continent, 'Unknown') AS region
FROM s
LEFT JOIN {catalog}.{schema}.bronze_country c
  ON lower(trim(s.country_clean)) = lower(trim(c.Country))
""")

print("silver tables built")
display(spark.sql(f"""
  SELECT region, COUNT(*) AS rows, ROUND(SUM(line_revenue),2) AS revenue
  FROM {catalog}.{schema}.silver_sales_enriched
  GROUP BY region ORDER BY revenue DESC
"""))